In [0]:
-- ============================================================
-- COMORBIDITY ANALYSIS — top co-occurring condition pairs
-- Self-join fact_conditions on patient_id to find disorders
-- that appear together in the same patient.
-- Scoped to condition_type = 'disorder' so social determinants
-- and administrative findings don't pollute the pairs.
-- code_a < code_b ensures each pair counted once, no self-pairs.
-- ============================================================
WITH patient_disorders AS (
  -- one distinct disorder per patient (a patient with 5 CKD
  -- episodes counts once for CKD)
  SELECT DISTINCT
    patient_id,
    condition_code,
    condition_desc
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_type = 'disorder'
),
pairs AS (
  SELECT
    a.condition_code AS code_a,
    a.condition_desc AS desc_a,
    b.condition_code AS code_b,
    b.condition_desc AS desc_b,
    a.patient_id
  FROM patient_disorders a
  JOIN patient_disorders b
    ON a.patient_id = b.patient_id
   AND a.condition_code < b.condition_code   -- unordered pairs, once each
)
SELECT
  desc_a,
  desc_b,
  COUNT(DISTINCT patient_id) AS patients_with_both
FROM pairs
GROUP BY desc_a, desc_b
ORDER BY patients_with_both DESC
LIMIT 25;

In [0]:
-- ============================================================
-- COMORBIDITY ANALYSIS — association strength via LIFT
-- Lift = P(A and B) / (P(A) * P(B))
--   lift = 1 -> independent (co-occur only as much as chance)
--   lift > 1 -> positively associated (genuine comorbidity)
--   lift >> 1 -> strong clinical link
-- This corrects the prevalence bias: common conditions no
-- longer dominate just for being common.
-- ============================================================
WITH patient_disorders AS (
  SELECT DISTINCT patient_id, condition_code, condition_desc
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_type = 'disorder'
),
total_patients AS (
  SELECT COUNT(DISTINCT patient_id) AS n_total
  FROM patient_disorders
),
-- prevalence of each single condition
prevalence AS (
  SELECT
    condition_code,
    condition_desc,
    COUNT(DISTINCT patient_id) AS n_with_condition
  FROM patient_disorders
  GROUP BY condition_code, condition_desc
),
-- co-occurrence counts per unordered pair
pairs AS (
  SELECT
    a.condition_code AS code_a,
    b.condition_code AS code_b,
    COUNT(DISTINCT a.patient_id) AS n_both
  FROM patient_disorders a
  JOIN patient_disorders b
    ON a.patient_id = b.patient_id
   AND a.condition_code < b.condition_code
  GROUP BY a.condition_code, b.condition_code
),
lift AS (
  SELECT
    pa.condition_desc AS condition_a,
    pb.condition_desc AS condition_b,
    p.n_both,
    pa.n_with_condition AS n_a,
    pb.n_with_condition AS n_b,
    t.n_total,
    -- lift = (n_both / n_total) / ((n_a / n_total) * (n_b / n_total))
    ROUND(
      (p.n_both * 1.0 * t.n_total) / (pa.n_with_condition * pb.n_with_condition),
      2
    ) AS lift
  FROM pairs p
  JOIN prevalence pa ON p.code_a = pa.condition_code
  JOIN prevalence pb ON p.code_b = pb.condition_code
  CROSS JOIN total_patients t
)
SELECT
  condition_a,
  condition_b,
  n_both,
  n_a,
  n_b,
  lift
FROM lift
WHERE n_both >= 30 -- floor out tiny-sample noise
ORDER BY lift DESC
LIMIT 30;